# RMT Assignment 4 — Kaggle runner

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2** (only one GPU is used — see the next cell). Never P100: timings must be comparable.
2. *Settings → Internet*: **On**.
3. *Add-ons → Secrets*: add `HF_TOKEN` (Hugging Face token with **write** access) and attach it to this notebook.
4. Edit the `CONFIG` cell below, then **Save Version → Save & Run All (Commit)** so the run continues with the browser closed (12 h limit per session).

Modes: `pilot` (≈20 min, checks memory, speed and the whole chain) → `lr_search` (Stage 1 only, dev data) → `run` (one final sequential run).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use exactly one T4 for every run
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ===== CONFIG =====
REPO_URL  = "https://github.com/<github-user>/rmt-peft-forgetting.git"
HUB_REPO  = "<hf-user>/rmt-peft-ckpts"   # private repo for checkpoints and logs (created automatically)
MODE      = "pilot"                        # "pilot" | "lr_search" | "run"

# MODE == "lr_search": which methods this account searches (can be split across accounts)
LR_METHODS = "full,lora,adapter"

# MODE == "run": one sequential run per session (see README, 'Run allocation')
METHOD = "lora"        # full | lora | adapter
SEED   = 1             # 1, 2, 3 (primary); 1 (replication, reversed order)
MODEL  = "primary"     # primary (Qwen2.5-0.5B) | replication (Qwen3-0.6B-Base)
ORDER  = "MLC"         # MLC = medicine -> law -> code ; CLM = reversed

In [ ]:
!git clone --depth 1 $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt
!git log -1 --format="commit %H  %s"

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

In [ ]:
# Adapter vs LoRA parameter matching on the real model configs
!python tests/check_param_matching.py --from_hub

In [ ]:
# Rebuild all data subsets; fails if they differ from the committed data/SHA256SUMS
!python src/prepare_data.py

In [ ]:
CKPT = "/kaggle/working/ckpts"
if MODE == "pilot":
    # 200 training examples per stage, 50 dev items: checks memory, speed and every code path.
    # Pilot numbers are NEVER reported as results.
    mid_lr = {"full": 2e-5, "lora": 2e-4, "adapter": 3e-4}
    for m in ["full", "lora", "adapter"]:
        !python src/train.py --method {m} --seed 1 --model primary --order MLC --lr {mid_lr[m]} \
            --train_n 200 --dev_n 50 --run_id pilot_{m} --out results/pilot/pilot_{m} \
            --ckpt_root {CKPT}/pilot --ckpt_fracs 0.5,1.0 --no_push_results --overwrite
        !rm -rf {CKPT}/pilot   # full fine-tuning checkpoints are ~1 GB each
elif MODE == "lr_search":
    !python src/lr_search.py --methods {LR_METHODS} --model primary --hub_repo {HUB_REPO}
elif MODE == "run":
    !python src/train.py --method {METHOD} --seed {SEED} --model {MODEL} --order {ORDER} \
        --ckpt_root {CKPT} --hub_repo {HUB_REPO} --delete_after_push
else:
    raise ValueError(MODE)

In [ ]:
# Summary of what this session produced (time, memory, dev metrics)
import json, glob
for f in sorted(glob.glob("results/**/run_summary.json", recursive=True)):
    s = json.load(open(f))
    print(s["run_id"])
    for st in s["stages"]:
        print(f"  stage {st['stage']} {st['domain']:9s} train {st['train_seconds']/60:6.1f} min  "
              f"dev {st['dev_eval_seconds']/60:5.1f} min  peak {st['peak_gpu_memory_gib']} GiB  "
              f"loss {st['final_train_loss']}")

In [ ]:
# Keep a downloadable copy of all logs (checkpoints are on the Hub)
!cd /kaggle/working && rm -f results.zip && zip -qr results.zip repo/results repo/data/manifest.json repo/data/SHA256SUMS
print("download /kaggle/working/results.zip from the Output tab")